# Trait prediction vs evaluation comparison

This notebook compares a prediction CSV against an evaluation CSV.

It:

- matches records by `key`
- compares comma-separated categorical values as unordered sets
- normalises case and whitespace
- supports canonicalisation such as `cypselae` → `cypsela`
- compares dimensional values numerically
- converts compatible units such as `6 cm` → `0.06 m` for a `[m]` trait
- scores partial matches at the individual-value level
- calculates unmatched-value counts and accuracy by taxon/row
- calculates unmatched-value counts and accuracy by trait/column
- creates a side-by-side Excel sheet with columns such as `habit_pred` and `habit_eval`
- highlights both cells red when prediction and evaluation differ
- outputs detailed mismatches to a separate sheet

Empty/empty trait cells are excluded from accuracy calculations.


In [1]:
import re
from pathlib import Path

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font
from openpyxl.utils import get_column_letter

In [2]:
DATA_DIR = Path('/Users/ben/Projects/ADEPT/Data')

OUTPUT_DIR = Path('/Users/ben/Projects/ADEPT/Code/PrePub/ADEPT/adept/data/processing/output')

ASSETS_DIR = Path('../adept/data/assets')



In [3]:
pred_file_path = OUTPUT_DIR / 'bhl/angiosperm-set123-54.bhl.traits.xlsx'
eval_file_path = DATA_DIR / 'ADEPT_eval_2309.xlsx'

eval_file_sheet_names=['set1', 'set2', 'set3']


In [4]:
traits_df = pd.read_parquet(ASSETS_DIR / 'traits.parquet')

In [5]:
eval_dfs = []

for sn in eval_file_sheet_names:
    eval_dfs.append(pd.read_excel(eval_file_path, sheet_name=sn))    
    
eval_df = pd.concat(eval_dfs)

# eval_df = pd.read_excel(eval_file_path, sheet_name=eval_file_sheet_name)
eval_df["taxon"] = eval_df["taxon"].astype("string").str.lower()

In [33]:
eval_df.to_csv(DATA_DIR / 'adept_trait_extraction_reference.csv', index=False)

In [7]:
unique_values = {}

for column in eval_df.columns:
    values = (
        eval_df[column]
        .dropna()
        .astype(str)
        .str.split(",")
        .explode()
        .str.strip()
    )
    unique_values[column] = sorted(values[values != ""].unique().tolist())

missing_chars = {}

all_chars = set(traits_df.character.unique())

for column, values in unique_values.items():

    if column in ['taxon', 'source', 'source_id', 'key'] or '[' in column: continue
    
    trait_chars = traits_df[traits_df['trait'] == column].character.unique()

    if trait_chars.any():

        difference = set(values) - set(trait_chars)
        if difference:
            missing_chars[column] = difference

    else:
        print(column)
        difference = set(values) - all_chars
        print(difference)
        


print(missing_chars)



life form
set()
calyx colour
set()
corolla colour
set()
corolla tube lipped
set()
stamen number
{'2.0', '4.0', '1.0', '5.0'}
stamenoid number
set()
carpel/ovary number
set()
heterostyly
set()
seeds per fruit min
set()
seeds per fruit max
{'2.0'}
seeds per fruit min.
{'2.0'}
ploidy level (2n)
{'120', '54', '2n = 14', '44', '2n = 8', '45', '12', '2n = 48', '2n=68', '2n = 54.', '28', '24', '2n = 38', '2n = 32', '40', '60', '2n = 18', '2n = 24', '100', '72', '2n = 22', '27', '30', '80', '36', '63', '2n = 16.', '58', '42', '56', '2n=32', '2n = 28'}
{}


In [8]:
eval_df["taxon"] 

0              achillea millefolium
1              achillea millefolium
2                 alchemilla alpina
3               andromeda polifolia
4                       betula nana
5                  carex capillaris
6                 cuscuta epithymum
7                   drosera anglica
8              eleocharis palustris
9                   galium palustre
10             hypochaeris radicata
11                lythrum salicaria
12                   montia fontana
13             phalaris arundinacea
14                    rumex acetosa
15              rumex conglomeratus
16                 salix reticulata
17                 stellaria alsine
18                    urtica dioica
19              vaccinium myrtillus
20                 valeriana dioica
0              agrostis stolonifera
1                 aira caryophyllea
2                     ajuga reptans
3             anthoxanthum odoratum
4             arrhenatherum elatius
5                     berula erecta
6                  caltha pa

In [28]:

pred_df_list = []

for source, group_df in eval_df.groupby("source"):
    df = pd.read_excel(pred_file_path, sheet_name=source)

    df["taxon"] = df["taxon"].astype("string").str.lower()


    if source == 'bhl':

        matched_df = df.merge(
            group_df[["taxon", "source_id"]].drop_duplicates(),
            on=["taxon", "source_id"],
            how="inner"
        )

    else:

        matched_df = df[df.taxon.isin(group_df.taxon)]

    pred_df_list.append(matched_df)

pred_df = pd.concat(pred_df_list, ignore_index=True)

# pred_df.taxon

pred_df


,taxon,life form,habitat,habit,clonality,perennial organ,plant min. height [m],plant max. height [m],indumentum,spinescence,...,seed max. length [mm],seed min. diameter [mm],seed max. diameter [mm],dispersal mode,ploidy level (2n),root min. depth [cm],root max. depth [cm],source,source_id,matched_name
0,rumex conglomeratus,NaN,NaN,NaN,NaN,NaN,0.025 m,0.05 m,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bhl,46694846.0,Rumex conglomeratus
1,mentha aquatica,perennial,terrestrial,erect leafy,"suckering, solitary plant",stem,NaN,NaN,pubescent,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bhl,22883976.0,Mentha aquatica; Mentha sativa
2,plantago lanceolata,perennial,terrestrial,erect leafy,rhizomatous,rhizome,NaN,NaN,NaN,NaN,...,4 mm,NaN,NaN,NaN,"2n = 12, 24",NaN,NaN,bhl,7934546.0,Plantago lanceolata
3,eleocharis palustris,perennial,terrestrial,"tussock, erect leafy, caespitose, scrambler","stoloniferous, solitary plant, rhizomatous","stolon, stem, rhizome",0.10 m,0.95 m,NaN,NaN,...,2.0 mm,NaN,NaN,ectozoochory,NaN,NaN,NaN,ecoflora,NaN,NaN
4,phalaris arundinacea,perennial,terrestrial,"erect leafy/tussock, scrambler","stoloniferous, rhizome","stolon, rhizome",0.60 m,2.00 m,glabrous,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ecoflora,NaN,NaN
5,ajuga reptans,perennial,terrestrial,"erect leafy, erect leafy/tussock","stoloniferous, solitary plant, rhizome","stolon, stem, rhizome",0.10 m,0.30 m,"glabrescent, glabrous",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ecoflora,NaN,NaN
6,campanula rotundifolia,perennial,terrestrial,"erect leafy, geoxylices, erect leafy/tussock","solitary plant, rhizome","rootstock, stem, rhizome",0.15 m,0.60 m,NaN,NaN,...,NaN,NaN,NaN,NaN,2n=68,NaN,NaN,ecoflora,NaN,NaN
7,alopecurus alpinus,perennial,terrestrial,"tussock, erect leafy/tussock, scrambler","stoloniferous, rhizomatous, rhizome","stolon, rhizome",0.10 m,0.45 m,"scabrous, glabrous",NaN,...,NaN,NaN,NaN,ectozoochory,NaN,NaN,NaN,ecoflora,NaN,NaN
8,ammophila arenaria,perennial,terrestrial,"erect leafy/tussock, scrambler","stoloniferous, rhizome","stolon, rhizome",0.50 m,1.20 m,pubescent,NaN,...,NaN,NaN,NaN,ectozoochory,NaN,NaN,NaN,ecoflora,NaN,NaN
9,bartsia alpina,perennial,terrestrial,erect leafy,solitary plant,stem,0.10 m,0.25 m,"glandular, pubescent",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ecoflora,NaN,NaN


In [29]:
def get_row_key(row):
    parts = [row.taxon, row.source]
    source_id = row.get("source_id")
    
    if pd.notna(source_id):   
        parts.append(str(source_id))
    
    return '|'.join(parts)

eval_df['key'] = eval_df.apply(get_row_key,  axis=1)    
pred_df['key'] = pred_df.apply(get_row_key,  axis=1)  

In [34]:
eval_df[eval_df['key'].isin(pred_df['key'])].to_csv(DATA_DIR / 'adept_trait_extraction_reference.csv', index=False)

pred_df.to_csv('evaluation_pred_dataset.csv', index=False)


# eval_df.to_csv(DATA_DIR / 'adept_trait_extraction_reference.csv', index=False)

In [12]:
eval_df['dispersal mode']

0              NaN
1       anemochory
2              NaN
3              NaN
4              NaN
5              NaN
6              NaN
7              NaN
8     ectozoochory
9              NaN
10    ectozoochory
11             NaN
12             NaN
13             NaN
14             NaN
15             NaN
16             NaN
17             NaN
18             NaN
19             NaN
20             NaN
0     ectozoochory
1     ectozoochory
2              NaN
3     ectozoochory
4     ectozoochory
5              NaN
6              NaN
7              NaN
8              NaN
9     ectozoochory
10    ectozoochory
11    ectozoochory
12    ectozoochory
13             NaN
14             NaN
15             NaN
16             NaN
17             NaN
18    ectozoochory
19             NaN
0              NaN
1              NaN
2              NaN
3     ectozoochory
4     ectozoochory
5              NaN
6              NaN
7              NaN
8     ectozoochory
9              NaN
10             NaN
11          

## Settings

In [13]:
PRED_FILE = "pred5.csv"
EVAL_FILE = "eval5.csv"
OUTPUT_FILE = "trait_evaluation.xlsx"

KEY_COL = "key"
TAXON_COL = "taxon"

META_COLS = {
    "key",
    "taxon",
    "source",
    "source_id",
    "description",
    "text",
    "Unnamed: 0",
}

CANONICAL_MAP = {
    "cypselae": "cypsela",
}

UNIT_TO_METRES = {
    "mm": 0.001,
    "cm": 0.01,
    "m": 1.0,
}

## Load data

In [31]:
# pred_df = pd.read_csv(PRED_FILE)
# eval_df = pd.read_csv(EVAL_FILE)

shared_keys = set(pred_df[KEY_COL]) & set(eval_df[KEY_COL])

print(f"Prediction rows: {len(pred_df):,}")
print(f"Evaluation rows: {len(eval_df):,}")
print(f"Shared keys: {len(shared_keys):,}")
print(f"Pred-only keys: {len(set(pred_df[KEY_COL]) - set(eval_df[KEY_COL])):,}")
print(f"Eval-only keys: {len(set(eval_df[KEY_COL]) - set(pred_df[KEY_COL])):,}")

Prediction rows: 50
Evaluation rows: 55
Shared keys: 50
Pred-only keys: 0
Eval-only keys: 5


## Normalisation helpers

In [15]:
def is_empty(value):
    """Return True for blank/NaN values."""
    return pd.isna(value) or str(value).strip() == ""


def canonicalise_text(value):
    """Normalise one categorical trait value."""
    value = str(value).strip().lower()
    value = re.sub(r"\s+", " ", value)
    return CANONICAL_MAP.get(value, value)


def get_column_unit(column):
    """
    Extract a dimensional unit from a column name.

    Examples:
        plant min. height [m] -> m
        seed max. length [mm] -> mm
    """
    match = re.search(r"\[(mm|cm|m)\]", column.lower())
    return match.group(1) if match else None


def normalise_measurement(token, target_unit):
    """
    Convert measurements to the unit specified by the trait column.

    Examples for a [m] column:
        "0.06 m" -> {0.06}
        "0.06"   -> {0.06}
        "6 cm"   -> {0.06}
    """
    if is_empty(token):
        return set()

    token = str(token).strip().lower()
    numbers = re.findall(r"-?\d+(?:\.\d+)?", token)

    if not numbers:
        return set()

    unit_match = re.search(r"\b(mm|cm|m)\b", token)
    source_unit = unit_match.group(1) if unit_match else target_unit

    result = set()

    for number in numbers:
        number = float(number)
        metres = number * UNIT_TO_METRES[source_unit]
        converted = metres / UNIT_TO_METRES[target_unit]
        result.add(round(converted, 10))

    return result


def normalise_cell(column, value):
    """
    Convert a trait cell into a set of normalised individual values.

    Thus:
        "ovate, lanceolate" == "lanceolate, ovate"

    and in a [m] column:
        "6 cm" == "0.06"
    """
    if is_empty(value):
        return set()

    unit = get_column_unit(column)

    if unit:
        return normalise_measurement(value, unit)

    return {
        canonicalise_text(v)
        for v in str(value).split(",")
        if v.strip()
    }

## Create side-by-side prediction/evaluation table

In [16]:
def create_side_by_side_comparison(pred_df, eval_df):
    """
    Create one row per shared key with prediction and evaluation
    values next to each other for every trait.

    Example columns:
        habit_pred
        habit_eval
        clonality_pred
        clonality_eval

    Returns:
        comparison_df
        mismatches
        trait_cols
    """

    trait_cols = [
        col
        for col in pred_df.columns.intersection(eval_df.columns)
        if col not in META_COLS
        and not col.startswith("Unnamed:")
    ]

    pred_indexed = pred_df.set_index(KEY_COL, drop=False)
    eval_indexed = eval_df.set_index(KEY_COL, drop=False)

    shared_keys = [
        key
        for key in eval_df[KEY_COL]
        if key in pred_indexed.index
    ]

    rows = []
    mismatches = set()

    for key in shared_keys:
        pred_row = pred_indexed.loc[key]
        eval_row = eval_indexed.loc[key]

        if isinstance(pred_row, pd.DataFrame):
            pred_row = pred_row.iloc[0]

        if isinstance(eval_row, pd.DataFrame):
            eval_row = eval_row.iloc[0]

        result = {
            "taxon": eval_row.get("taxon", ""),
            "source": eval_row.get("source", ""),
            "key": key,
        }

        for trait in trait_cols:
            pred_value = pred_row.get(trait)
            eval_value = eval_row.get(trait)

            pred_col = f"{trait}_pred"
            eval_col = f"{trait}_eval"

            result[pred_col] = pred_value
            result[eval_col] = eval_value

            pred_norm = normalise_cell(trait, pred_value)
            eval_norm = normalise_cell(trait, eval_value)

            if pred_norm != eval_norm:
                mismatches.add((key, pred_col, eval_col))

        rows.append(result)

    return pd.DataFrame(rows), mismatches, trait_cols


comparison_df, comparison_mismatches, trait_cols = (
    create_side_by_side_comparison(pred_df, eval_df)
)

comparison_df.head()

,taxon,source,key,life form_pred,life form_eval,habitat_pred,habitat_eval,habit_pred,habit_eval,clonality_pred,...,seed max. diameter [mm]_pred,seed max. diameter [mm]_eval,dispersal mode_pred,dispersal mode_eval,ploidy level (2n)_pred,ploidy level (2n)_eval,root min. depth [cm]_pred,root min. depth [cm]_eval,root max. depth [cm]_pred,root max. depth [cm]_eval
0,achillea millefolium,efloras.flora_of_china,achillea millefolium|efloras.flora_of_china,perennial,perennial,terrestrial,terrestrial,"herb, erect leafy, erect leafy/tussock","herb, erect leafy/tussock, erect leafy","solitary plant, rhizome",...,NaN,NaN,NaN,NaN,"2n = 18, 27, 36","2n = 18, 27, 36",NaN,NaN,NaN,NaN
1,achillea millefolium,efloras.flora_of_north_america,achillea millefolium|efloras.flora_of_north_am...,perennial,perennial,terrestrial,terrestrial,"stoloniferous, erect leafy/tussock, erect leafy","erect leafy/tussock, erect leafy, stoloniferous","stoloniferous, solitary plant, rhizomatous, rh...",...,NaN,NaN,anemochory,anemochory,"2n = 18, 27, 36, 45, 54, 63, 72","2n = 18, 27, 36, 45, 54, 63, 72",NaN,NaN,NaN,NaN
2,alchemilla alpina,efloras.flora_of_north_america,alchemilla alpina|efloras.flora_of_north_america,NaN,NaN,terrestrial,terrestrial,erect leafy,erect leafy,solitary plant,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,andromeda polifolia,efloras.flora_of_north_america,andromeda polifolia|efloras.flora_of_north_ame...,NaN,NaN,terrestrial,terrestrial,"geoxylices, scrambler","geoxylices, scrambler",stoloniferous,...,NaN,NaN,NaN,NaN,2n = 48,2n = 48,NaN,NaN,NaN,NaN
4,betula nana,efloras.flora_of_north_america,betula nana|efloras.flora_of_north_america,NaN,NaN,terrestrial,terrestrial,"tree/shrub, climber/scrambler, shrub, scrambler","shrub, tree/shrub, climber/scrambler, scrambler","stoloniferous, solitary plant",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Calculate row, trait and detailed mismatch summaries

In [17]:
def evaluate_traits(pred_df, eval_df, trait_cols):
    """
    Compare every individual trait value for all shared rows.

    Value accuracy:
        matched / (matched + extra predicted + missed eval)

    Empty/empty trait cells are ignored.
    """

    pred_indexed = pred_df.set_index(KEY_COL, drop=False)
    eval_indexed = eval_df.set_index(KEY_COL, drop=False)

    shared_keys = [
        key
        for key in eval_df[KEY_COL]
        if key in pred_indexed.index
    ]

    row_results = []
    differences = []

    trait_totals = {
        trait: {
            "matched": 0,
            "extra_pred": 0,
            "missed_eval": 0,
            "exact_cells": 0,
            "different_cells": 0,
            "populated_cells": 0,
        }
        for trait in trait_cols
    }

    for key in shared_keys:
        pred_row = pred_indexed.loc[key]
        eval_row = eval_indexed.loc[key]

        if isinstance(pred_row, pd.DataFrame):
            pred_row = pred_row.iloc[0]

        if isinstance(eval_row, pd.DataFrame):
            eval_row = eval_row.iloc[0]

        taxon = eval_row.get(TAXON_COL, key)
        source = eval_row.get("source", "")

        row_matched = 0
        row_extra = 0
        row_missed = 0
        exact_trait_cells = 0
        different_trait_cells = 0
        populated_trait_cells = 0

        for trait in trait_cols:
            pred_raw = pred_row.get(trait)
            eval_raw = eval_row.get(trait)

            pred_values = normalise_cell(trait, pred_raw)
            eval_values = normalise_cell(trait, eval_raw)

            if not pred_values and not eval_values:
                continue

            populated_trait_cells += 1
            trait_totals[trait]["populated_cells"] += 1

            matched = pred_values & eval_values
            extra_pred = pred_values - eval_values
            missed_eval = eval_values - pred_values

            n_matched = len(matched)
            n_extra = len(extra_pred)
            n_missed = len(missed_eval)

            row_matched += n_matched
            row_extra += n_extra
            row_missed += n_missed

            trait_totals[trait]["matched"] += n_matched
            trait_totals[trait]["extra_pred"] += n_extra
            trait_totals[trait]["missed_eval"] += n_missed

            if pred_values == eval_values:
                exact_trait_cells += 1
                trait_totals[trait]["exact_cells"] += 1
            else:
                different_trait_cells += 1
                trait_totals[trait]["different_cells"] += 1

                differences.append({
                    "taxon": taxon,
                    "source": source,
                    "key": key,
                    "trait": trait,
                    "Pred": pred_raw,
                    "Eval": eval_raw,
                    "matched values": ", ".join(map(str, sorted(matched))),
                    "extra predicted values": ", ".join(
                        map(str, sorted(extra_pred))
                    ),
                    "missed eval values": ", ".join(
                        map(str, sorted(missed_eval))
                    ),
                    "matched count": n_matched,
                    "extra pred count": n_extra,
                    "missed eval count": n_missed,
                })

        unmatched = row_extra + row_missed
        total_union = row_matched + unmatched

        value_accuracy = (
            100 * row_matched / total_union
            if total_union
            else 100.0
        )

        cell_accuracy = (
            100 * exact_trait_cells / populated_trait_cells
            if populated_trait_cells
            else 100.0
        )

        row_results.append({
            "taxon": taxon,
            "source": source,
            "key": key,
            "matched values": row_matched,
            "extra predicted values": row_extra,
            "missed eval values": row_missed,
            "unmatched values": unmatched,
            "total values": total_union,
            "value accuracy %": value_accuracy,
            "exact trait cells": exact_trait_cells,
            "different trait cells": different_trait_cells,
            "populated trait cells": populated_trait_cells,
            "trait cell accuracy %": cell_accuracy,
        })

    trait_results = []

    for trait, counts in trait_totals.items():
        if counts["populated_cells"] == 0:
            continue

        matched = counts["matched"]
        extra = counts["extra_pred"]
        missed = counts["missed_eval"]
        unmatched = extra + missed
        total = matched + unmatched

        value_accuracy = (
            100 * matched / total
            if total
            else 100.0
        )

        cell_accuracy = (
            100 * counts["exact_cells"] / counts["populated_cells"]
        )

        trait_results.append({
            "trait": trait,
            "matched values": matched,
            "extra predicted values": extra,
            "missed eval values": missed,
            "unmatched values": unmatched,
            "total values": total,
            "value accuracy %": value_accuracy,
            "exact cells": counts["exact_cells"],
            "different cells": counts["different_cells"],
            "populated cells": counts["populated_cells"],
            "trait cell accuracy %": cell_accuracy,
        })

    return (
        pd.DataFrame(row_results),
        pd.DataFrame(trait_results),
        pd.DataFrame(differences),
    )


row_summary, trait_summary, differences = evaluate_traits(
    pred_df,
    eval_df,
    trait_cols,
)

## Row-level summary

In [18]:
row_summary.sort_values(
    "value accuracy %"
).reset_index(drop=True)

,taxon,source,key,matched values,extra predicted values,missed eval values,unmatched values,total values,value accuracy %,exact trait cells,different trait cells,populated trait cells,trait cell accuracy %
0,cardamine flexuosa,efloras.flora_of_north_america,cardamine flexuosa|efloras.flora_of_north_america,18,10,18,28,46,39.130435,8,19,27,29.629630
1,eriophorum vaginatum,efloras.flora_of_north_america,eriophorum vaginatum|efloras.flora_of_north_am...,11,6,11,17,28,39.285714,8,13,21,38.095238
2,urtica dioica,efloras.flora_of_pakistan,urtica dioica|efloras.flora_of_pakistan,15,2,19,21,36,41.666667,12,13,25,48.000000
3,plantago lanceolata,bhl,plantago lanceolata|bhl|7934546.0,15,8,8,16,31,48.387097,9,15,24,37.500000
4,carex capillaris,efloras.flora_of_north_america,carex capillaris|efloras.flora_of_north_america,13,2,7,9,22,59.090909,13,8,21,61.904762
5,berula erecta,efloras.flora_of_china,berula erecta|efloras.flora_of_china,13,0,9,9,22,59.090909,10,8,18,55.555556
6,erica vagans,efloras.flora_of_north_america,erica vagans|efloras.flora_of_north_america,22,6,7,13,35,62.857143,19,11,30,63.333333
7,cardamine pratensis,efloras.flora_of_north_america,cardamine pratensis|efloras.flora_of_north_ame...,26,2,12,14,40,65.000000,21,6,27,77.777778
8,eriophorum angustifolium,efloras.flora_of_north_america,eriophorum angustifolium|efloras.flora_of_nort...,19,1,9,10,29,65.517241,13,10,23,56.521739
9,achillea millefolium,efloras.flora_of_china,achillea millefolium|efloras.flora_of_china,27,5,9,14,41,65.853659,15,9,24,62.500000


## Trait-level summary

In [19]:
trait_summary.sort_values(
    "value accuracy %"
).reset_index(drop=True)

,trait,matched values,extra predicted values,missed eval values,unmatched values,total values,value accuracy %,exact cells,different cells,populated cells,trait cell accuracy %
0,seeds per fruit min.,0,0,1,1,1,0.0,0,1,1,0.0
1,seed min. diameter [mm],0,0,1,1,1,0.0,0,1,1,0.0
2,reproduction architecture,0,2,0,2,2,0.0,0,2,2,0.0
3,seeds per fruit max,0,0,1,1,1,0.0,0,1,1,0.0
4,flower shape,0,0,2,2,2,0.0,0,2,2,0.0
...,...,...,...,...,...,...,...,...,...,...,...
58,flower orientation,5,0,0,0,5,100.0,5,0,5,100.0
59,flower symmetry,1,0,0,0,1,100.0,1,0,1,100.0
60,inflorescence orientation,5,0,0,0,5,100.0,5,0,5,100.0
61,leaf arrangement,5,0,0,0,5,100.0,5,0,5,100.0


## Detailed differences

In [20]:
differences.head(50)

,taxon,source,key,trait,Pred,Eval,matched values,extra predicted values,missed eval values,matched count,extra pred count,missed eval count
0,achillea millefolium,efloras.flora_of_china,achillea millefolium|efloras.flora_of_china,indumentum,"glandular, villous, villouse","glandular, bristly",glandular,"villous, villouse",bristly,1,2,1
1,achillea millefolium,efloras.flora_of_china,achillea millefolium|efloras.flora_of_china,leaf architecture,"pinnatisect, pinnatipartite",pinnatisect,pinnatisect,pinnatipartite,,1,1,0
2,achillea millefolium,efloras.flora_of_china,achillea millefolium|efloras.flora_of_china,leaf apex,"attenuate, lanceolate",attenuate,attenuate,lanceolate,,1,1,0
3,achillea millefolium,efloras.flora_of_china,achillea millefolium|efloras.flora_of_china,inflorescence arrangement,NaN,"panicle, capitulum",,,"capitulum, panicle",0,0,2
4,achillea millefolium,efloras.flora_of_china,achillea millefolium|efloras.flora_of_china,flower shape,NaN,tubulate,,,tubulate,0,0,1
5,achillea millefolium,efloras.flora_of_china,achillea millefolium|efloras.flora_of_china,flower colour,NaN,"white, pink, purple",,,"pink, purple, white",0,0,3
6,achillea millefolium,efloras.flora_of_china,achillea millefolium|efloras.flora_of_china,fruit type,NaN,achene,,,achene,0,0,1
7,achillea millefolium,efloras.flora_of_china,achillea millefolium|efloras.flora_of_china,seed min. width [mm],NaN,2.0,,,2.0,0,0,1
8,achillea millefolium,efloras.flora_of_china,achillea millefolium|efloras.flora_of_china,seed max. length [mm],2 mm,NaN,,2.0,,0,1,0
9,achillea millefolium,efloras.flora_of_north_america,achillea millefolium|efloras.flora_of_north_am...,fruit type,NaN,cypsela,,,cypsela,0,0,1


## Overall precision, recall and F1

In [21]:
matched = row_summary["matched values"].sum()
extra = row_summary["extra predicted values"].sum()
missed = row_summary["missed eval values"].sum()

precision = matched / (matched + extra) if (matched + extra) else 0
recall = matched / (matched + missed) if (matched + missed) else 0
f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall)
    else 0
)

print(f"Matched values: {matched:,}")
print(f"Extra predicted values: {extra:,}")
print(f"Missed eval values: {missed:,}")
print(f"Precision: {precision:.1%}")
print(f"Recall: {recall:.1%}")
print(f"F1: {f1:.1%}")

Matched values: 1,078
Extra predicted values: 116
Missed eval values: 221
Precision: 90.3%
Recall: 83.0%
F1: 86.5%


## Write Excel workbook

In [22]:
def format_worksheet(ws):
    """Apply simple readable formatting to a worksheet."""

    header_fill = PatternFill(
        fill_type="solid",
        fgColor="D9EAF7",
    )
    header_font = Font(bold=True)

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font

    for column_cells in ws.columns:
        column_letter = get_column_letter(column_cells[0].column)

        max_length = 0

        for cell in column_cells:
            if cell.value is not None:
                max_length = max(
                    max_length,
                    len(str(cell.value)),
                )

        ws.column_dimensions[column_letter].width = min(
            max(max_length + 2, 10),
            45,
        )


def create_excel_report(
    comparison_df,
    comparison_mismatches,
    row_summary,
    trait_summary,
    differences,
    output_file,
):
    """
    Write the full Excel comparison workbook.

    Sheets:
      - Side by side
      - Row summary
      - Trait summary
      - Differences
    """

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        comparison_df.to_excel(
            writer,
            sheet_name="Side by side",
            index=False,
        )

        row_summary.to_excel(
            writer,
            sheet_name="Row summary",
            index=False,
        )

        trait_summary.to_excel(
            writer,
            sheet_name="Trait summary",
            index=False,
        )

        differences.to_excel(
            writer,
            sheet_name="Differences",
            index=False,
        )

    wb = load_workbook(output_file)

    red_fill = PatternFill(
        fill_type="solid",
        fgColor="FFC7CE",
    )
    red_font = Font(color="9C0006")

    ws = wb["Side by side"]

    headers = {
        cell.value: cell.column
        for cell in ws[1]
    }

    key_column = headers[KEY_COL]

    key_to_row = {}

    for excel_row in range(2, ws.max_row + 1):
        key = ws.cell(
            row=excel_row,
            column=key_column,
        ).value

        key_to_row[key] = excel_row

    for key, pred_col, eval_col in comparison_mismatches:
        if key not in key_to_row:
            continue

        row_number = key_to_row[key]

        for column_name in (pred_col, eval_col):
            if column_name not in headers:
                continue

            cell = ws.cell(
                row=row_number,
                column=headers[column_name],
            )

            cell.fill = red_fill
            cell.font = red_font

    for ws in wb.worksheets:
        format_worksheet(ws)

    for sheet_name in ["Row summary", "Trait summary"]:
        ws = wb[sheet_name]

        for header_cell in ws[1]:
            if "%" in str(header_cell.value):
                for row in range(2, ws.max_row + 1):
                    ws.cell(
                        row=row,
                        column=header_cell.column,
                    ).number_format = "0.0"

    wb.save(output_file)


create_excel_report(
    comparison_df=comparison_df,
    comparison_mismatches=comparison_mismatches,
    row_summary=row_summary,
    trait_summary=trait_summary,
    differences=differences,
    output_file=OUTPUT_FILE,
)

print(f"Excel report written to: {Path(OUTPUT_FILE).resolve()}")

Excel report written to: /Users/ben/Projects/ADEPT/Code/PrePub/ADEPT/notebooks/trait_evaluation.xlsx


## Optional: inspect the worst-performing taxa

In [23]:
row_summary[
    [
        "taxon",
        "source",
        "unmatched values",
        "value accuracy %",
        "trait cell accuracy %",
    ]
].sort_values(
    "value accuracy %"
).reset_index(drop=True)

,taxon,source,unmatched values,value accuracy %,trait cell accuracy %
0,cardamine flexuosa,efloras.flora_of_north_america,28,39.130435,29.629630
1,eriophorum vaginatum,efloras.flora_of_north_america,17,39.285714,38.095238
2,urtica dioica,efloras.flora_of_pakistan,21,41.666667,48.000000
3,plantago lanceolata,bhl,16,48.387097,37.500000
4,carex capillaris,efloras.flora_of_north_america,9,59.090909,61.904762
5,berula erecta,efloras.flora_of_china,9,59.090909,55.555556
6,erica vagans,efloras.flora_of_north_america,13,62.857143,63.333333
7,cardamine pratensis,efloras.flora_of_north_america,14,65.000000,77.777778
8,eriophorum angustifolium,efloras.flora_of_north_america,10,65.517241,56.521739
9,achillea millefolium,efloras.flora_of_china,14,65.853659,62.500000


## Optional: inspect the worst-performing traits

In [24]:
trait_summary[
    [
        "trait",
        "unmatched values",
        "extra predicted values",
        "missed eval values",
        "value accuracy %",
        "trait cell accuracy %",
    ]
].sort_values(
    "value accuracy %"
).reset_index(drop=True)

,trait,unmatched values,extra predicted values,missed eval values,value accuracy %,trait cell accuracy %
0,seeds per fruit min.,1,0,1,0.0,0.0
1,seed min. diameter [mm],1,0,1,0.0,0.0
2,reproduction architecture,2,2,0,0.0,0.0
3,seeds per fruit max,1,0,1,0.0,0.0
4,flower shape,2,0,2,0.0,0.0
...,...,...,...,...,...,...
58,flower orientation,0,0,0,100.0,100.0
59,flower symmetry,0,0,0,100.0,100.0
60,inflorescence orientation,0,0,0,100.0,100.0
61,leaf arrangement,0,0,0,100.0,100.0


## Optional: inspect differences for one taxon

In [25]:
taxon_search = "Achillea millefolium"

differences[
    differences["taxon"].str.contains(
        taxon_search,
        case=False,
        na=False,
    )
].reset_index(drop=True)

,taxon,source,key,trait,Pred,Eval,matched values,extra predicted values,missed eval values,matched count,extra pred count,missed eval count
0,achillea millefolium,efloras.flora_of_china,achillea millefolium|efloras.flora_of_china,indumentum,"glandular, villous, villouse","glandular, bristly",glandular,"villous, villouse",bristly,1,2,1
1,achillea millefolium,efloras.flora_of_china,achillea millefolium|efloras.flora_of_china,leaf architecture,"pinnatisect, pinnatipartite",pinnatisect,pinnatisect,pinnatipartite,,1,1,0
2,achillea millefolium,efloras.flora_of_china,achillea millefolium|efloras.flora_of_china,leaf apex,"attenuate, lanceolate",attenuate,attenuate,lanceolate,,1,1,0
3,achillea millefolium,efloras.flora_of_china,achillea millefolium|efloras.flora_of_china,inflorescence arrangement,NaN,"panicle, capitulum",,,"capitulum, panicle",0,0,2
4,achillea millefolium,efloras.flora_of_china,achillea millefolium|efloras.flora_of_china,flower shape,NaN,tubulate,,,tubulate,0,0,1
5,achillea millefolium,efloras.flora_of_china,achillea millefolium|efloras.flora_of_china,flower colour,NaN,"white, pink, purple",,,"pink, purple, white",0,0,3
6,achillea millefolium,efloras.flora_of_china,achillea millefolium|efloras.flora_of_china,fruit type,NaN,achene,,,achene,0,0,1
7,achillea millefolium,efloras.flora_of_china,achillea millefolium|efloras.flora_of_china,seed min. width [mm],NaN,2.0,,,2.0,0,0,1
8,achillea millefolium,efloras.flora_of_china,achillea millefolium|efloras.flora_of_china,seed max. length [mm],2 mm,NaN,,2.0,,0,1,0
9,achillea millefolium,efloras.flora_of_north_america,achillea millefolium|efloras.flora_of_north_am...,fruit type,NaN,cypsela,,,cypsela,0,0,1
